# Gerar pela API ou conferir uma proposta

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="02_teste_inferencia_planilha_turmas"


1. Preflight e seleção. READ_EXISTING não escreve; GENERATE_AND_VALIDATE envia POST ao worker. Escolha explicitamente snapshot e modelo/versão; confirme a escrita e defina request_key.

In [ ]:
from src.workbench.contract_review import preflight,candidates,fetch_payload,reconcile
display(pd.DataFrame(preflight(cfg)));display(pd.DataFrame(candidates(cfg)))
proposal_id=str(params.proposal_id) if params.proposal_id else ""


2. Submeter geração somente no modo opt-in. Nenhuma persistência é feita pelo notebook. Esperado: job, correlation_id e status_url.

In [ ]:
if params.mode=="GENERATE_AND_VALIDATE":
    params.authorize()
    if not params.snapshot_id or not params.model_version:
        raise ValueError("Selecione snapshot_id e model_version explicitamente")
    from src.workbench.operations_sdk import generate
    payload=dict(cycle_code=cfg.cycle,snapshot_id=str(params.snapshot_id),
        model_selectors=[dict(model_name=params.model_name,model_version=params.model_version)],
        capacity=params.capacity,requested_by=params.requester,confirm_demo=True)
    display(payload)
    submitted=generate(cfg,payload,params.request_key)
    display(submitted)
    finished=await_job(cfg,submitted["status_url"])
    if finished["status"]!="COMPLETED":
        raise RuntimeError("Geração não completou; consulte o job persistido: "+str(finished))
    proposal_id=finished["proposal_id"]
    evidence.update(request=payload,job=finished)
if not proposal_id:
    raise ValueError("Selecione proposal_id da tabela ou configure modo GENERATE_AND_VALIDATE")


3. Obter payload integral paginado e validar schema/hash. Este é o JSON consumido pela aplicação.

In [ ]:
pages,offers=fetch_payload(cfg,proposal_id,page_size=50)
for page in pages: display(page)
display(pd.json_normalize(offers))
display(pd.DataFrame(pages[0]["validation"]))
assert not any(r["status"]=="FAIL" for r in pages[0]["validation"])
evidence.update(proposal_id=proposal_id,http_pages=pages)


4. Reconciliação somente leitura: linhas SQL, versões Registry, snapshot e bytes RustFS. Esperado: mesmos IDs, quantidades e hashes.

In [ ]:
evidence["reconciliation"]=reconcile(cfg,proposal_id,pages,offers)
display(evidence["reconciliation"])
display(Markdown(f"[API](http://localhost:8000/v1/proposals/{proposal_id}/application-view) · [MLflow]({cfg.ui}) · [RustFS](http://localhost:9001)"))


Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [ ]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
display(report(NOTEBOOK_ID,evidence,status=report_status,limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))
